In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
data = pd.read_csv('../data/crop_recommendation.csv')
data

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice
...,...,...,...,...,...,...,...,...
2195,107,34,32,26.774637,66.413269,6.780064,177.774507,coffee
2196,99,15,27,27.417112,56.636362,6.086922,127.924610,coffee
2197,118,33,30,24.131797,67.225123,6.362608,173.322839,coffee
2198,117,32,34,26.272418,52.127394,6.758793,127.175293,coffee


### Regresión logística 

Modelo clasificador basado en regresión logística.
Utiliza penalización Elastic-Net optimizada con el algoritmo saga.
Se define su arquitectura con un pipeline que incluye el escalado de datos, previniendo fuga de datos al realizar la validación cruzada con GridSearchCV.

Se alcanza un accuracy del 97.5%

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [12]:
# definir variables predictoras y variable objetivo

X = data.drop(columns=['label'])
y = data.label
X.shape, y.shape

((2200, 7), (2200,))

In [13]:
# definir pipeline y parámetros para GridSearchCV

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

params = {
        "model__solver": ["saga"],  # el solver saga es el único que soporta elasticnet!
        "model__penalty": ["elasticnet"],  #penalty fue deprecado en 1.8 y eliminado en 1.10 (estamos usando 1.9)
        "model__l1_ratio": [0, 0.3, 0.5, 0.7, 1],
        "model__max_iter": [2000, 5000],
        "model__C": [0.1, 1, 10, 100],
    }


## Cross Val con  'accuracy' como score para buscar hiperparámetros

In [ ]:
# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Grid Search para encontrar los mejores parámetros del modelo
grid_search = GridSearchCV(pipeline, params, cv=5, scoring="roc_auc_ovr", n_jobs=12, verbose = 1)
grid_search.fit(X_train, y_train)
display(grid_search.best_params_)
display(grid_search.best_score_)
pipeline = grid_search.best_estimator_

"""

{'model__C': 10,
 'model__l1_ratio': 1,
 'model__max_iter': 2000,
 'model__penalty': 'elasticnet',
 'model__solver': 'saga'}
 
 np.float64(0.9821318674705756)
"""

In [15]:
from sklearn.metrics import classification_report, accuracy_score
#predicciones
y_pred = pipeline.predict(X_test)

#mostrar resultados
print(classification_report(y_test, y_pred))

#accuracy 
print("Accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        23
      banana       1.00      1.00      1.00        21
   blackgram       0.90      0.95      0.93        20
    chickpea       1.00      1.00      1.00        26
     coconut       1.00      1.00      1.00        27
      coffee       0.94      1.00      0.97        17
      cotton       1.00      1.00      1.00        17
      grapes       1.00      1.00      1.00        14
        jute       0.87      0.87      0.87        23
 kidneybeans       0.95      1.00      0.98        20
      lentil       0.92      1.00      0.96        11
       maize       1.00      1.00      1.00        21
       mango       1.00      1.00      1.00        19
   mothbeans       1.00      1.00      1.00        24
    mungbean       1.00      1.00      1.00        19
   muskmelon       1.00      1.00      1.00        17
      orange       1.00      1.00      1.00        14
      papaya       0.96    


## Resultados



| Scoring en CV | Desempeño promedio en CV | Accuracy obtenida |
| ---------------- | ---------------- | -------- |
| Accuracy | 0.9823 | 0.970  |
| f1_macro | 0.9821 | 0.970 | 
| roc_auc_ovr | 0.9998 | 0.975 |